# cross-entropy-classification-loss — worked example 1: Cross-entropy against one-hot soft targets

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-entropy-classification-loss`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn.functional as F

## Concept

Cross-entropy on logits is `-(target_probs * log_softmax(logits)).sum(dim=-1).mean()`. When the targets are hard class indices, the target distribution is one-hot, so the sum collapses to picking out the log-prob of the correct class. Verifying the soft-target form against `F.cross_entropy` on hard labels confirms the two are identical in the one-hot case.

## Worked solution

We want to show the general (soft-target) cross-entropy reduces to the standard hard-label loss when targets are one-hot.

1. **Stable log-probs.** `log_probs = F.log_softmax(logits, dim=-1)` gives `(B, C)`. Using `log_softmax` rather than `log(softmax(...))` avoids overflow in the exponentials — this matters because the inputs are raw logits, not probabilities.
2. **Build the one-hot target matrix.** `targets = F.one_hot(labels, num_classes=C).float()` is `(B, C)` with a single 1.0 per row at the correct class. This is the explicit probability distribution each example's loss is measured against.
3. **Element-wise cross-entropy.** `-(targets * log_probs)` zeroes out every entry except the correct class, so summing over the class axis (`.sum(dim=-1)`) yields the per-example NLL `(B,)`. The one-hot mask is *why* the general formula collapses to gathering the target log-prob.
4. **Reduce.** `.mean()` averages over the batch, matching `F.cross_entropy`'s default `reduction='mean'`.

The final assert confirms it equals `F.cross_entropy(logits, labels)` to 1e-5.

In [ ]:
import torch.nn.functional as F

def soft_target_ce(logits, labels):
    C = logits.shape[-1]
    log_probs = F.log_softmax(logits, dim=-1)
    targets = F.one_hot(labels, num_classes=C).float()
    per_ex = -(targets * log_probs).sum(dim=-1)
    return per_ex.mean()

t.manual_seed(0)
logits = t.randn(5, 4)
labels = t.tensor([2, 0, 3, 1, 2])
mine = soft_target_ce(logits, labels)
ref = F.cross_entropy(logits, labels)
print("mine:", round(mine.item(), 6), "ref:", round(ref.item(), 6))
print("match:", t.allclose(mine, ref, atol=1e-5))